# CSIRO Biomass — Unified Submission Notebook

Submit **any of the 5 trained models** by changing a single variable: `CHOSEN_MODEL`.

Because preprocessing (`Rescaling(255) -> preprocess_input`) is baked **inside** each saved
model, the inference code is identical for every backbone — only the model file, its best
stage, and the ViT dependency differ (handled by the registry below).

**How to use**
1. Add each trained model's output as a Kaggle *input dataset* and fix its path in `MODEL_REGISTRY`.
2. Set `CHOSEN_MODEL` to one of the five constants.
3. Run all cells -> `submission.csv`.

Key facts baked into this notebook:
- Models output **RAW grams** (`USE_LOG_TARGETS = False`) -> **NO `expm1`**.
- Feed images as **[0,1]** float, 224x224 (model rescales internally).
- `GDM` / `Total` are auto-derived; the GDM formula is **auto-detected** from `train.csv`.

## 1. Choose the model

In [ ]:
# =========================================================
# 1. MODEL SELECTION  --  the ONLY things you normally edit
# =========================================================

# --- the five model constants ---
MOBILENET      = "mobilenet"
EFFICIENTNETB0 = "efficientnetb0"
EFFICIENTNETB3 = "efficientnetb3"
DENSENET121    = "densenet121"
VIT_B16        = "vit_b16"

# >>> pick which model to submit <<<
CHOSEN_MODEL = VIT_B16      # e.g. EFFICIENTNETB0, DENSENET121, MOBILENET, EFFICIENTNETB3, VIT_B16

# >>> path to the uploaded `models` folder (Kaggle dataset) <<<
# Upload Thesis/models/ as a Kaggle dataset and set this to its mount path.
# If files land directly under the dataset root, drop the trailing "/models".
MODELS_DIR = "/kaggle/input/datasets/phylosopherhossain/csiro-biomass-models"

# --- per-model registry --------------------------------------------------
# filename        : the best-stage model copied into models/ (main for EfficientNets, ft for the rest)
# best_stage      : which training stage that file is (info only; see model_comparison.md)
# needs_vit_keras : True only for ViT (custom layers require vit-keras + tensorflow-addons)
MODEL_REGISTRY = {
    MOBILENET: {
        "name": "MobileNetV0",   "filename": "mobilenet.keras",
        "best_stage": "ft",      "needs_vit_keras": False, "weighted_r2": 0.6463,
    },
    EFFICIENTNETB0: {
        "name": "EfficientNetB0","filename": "efficientnetb0.keras",
        "best_stage": "main",    "needs_vit_keras": False, "weighted_r2": 0.6942,
    },
    EFFICIENTNETB3: {
        "name": "EfficientNetB3","filename": "efficientnetb3.keras",
        "best_stage": "main",    "needs_vit_keras": False, "weighted_r2": 0.6831,
    },
    DENSENET121: {
        "name": "DenseNet121",   "filename": "densenet121.keras",
        "best_stage": "ft",      "needs_vit_keras": False, "weighted_r2": 0.6812,
    },
    VIT_B16: {
        "name": "ViT-B16",       "filename": "vitb16.keras",
        "best_stage": "ft",      "needs_vit_keras": True,  "weighted_r2": 0.7356,
    },
}

CFG = MODEL_REGISTRY[CHOSEN_MODEL]
CFG["model_path"] = f"{MODELS_DIR}/{CFG['filename']}"
print(f"Selected model : {CFG['name']}  (best stage: {CFG['best_stage']}, val weighted R2 ~ {CFG['weighted_r2']})")
print(f"Model file     : {CFG['model_path']}")

## 2. Dependencies (ViT only)

In [ ]:
# =========================================================
# 2. INSTALL ViT DEPENDENCY  (only if the chosen model needs it)
# =========================================================
# IMPORTANT: install ONLY vit-keras. Do NOT install tensorflow-addons --
# it is broken on TF 2.16+ / Keras 3 (import crashes with
# "No module named 'keras.src.engine'") and vit-keras does not require it.
if CFG["needs_vit_keras"]:
    import subprocess, sys
    print("Installing vit-keras for ViT ...")
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "vit-keras"])
    print("Done.")
else:
    print("No extra dependencies needed for", CFG["name"])

## 3. Imports & paths

In [ ]:
# =========================================================
# 3. IMPORTS & PATHS
# =========================================================
import os
import numpy as np
import pandas as pd
import tensorflow as tf

KAGGLE_PATH     = "/kaggle/input/csiro-biomass"
TRAIN_CSV_PATH  = os.path.join(KAGGLE_PATH, "train.csv")
TEST_CSV_PATH   = os.path.join(KAGGLE_PATH, "test.csv")
SAMPLE_SUB_PATH = os.path.join(KAGGLE_PATH, "sample_submission.csv")

IMG_SIZE   = 224
BATCH_SIZE = 32

# The 3 base targets the model predicts, in output order:
BASE_TARGETS = ["Dry_Green_g", "Dry_Dead_g", "Dry_Clover_g"]
print("TF version:", tf.__version__)

## 4. Load the chosen model

In [ ]:
# =========================================================
# 4. LOAD MODEL  (compile=False -> no need for the custom loss)
# =========================================================
import glob, importlib

# For ViT, importing vit_keras registers its custom layers so the model deserializes.
if CFG["needs_vit_keras"]:
    from vit_keras import vit                # noqa: F401  (registers ViT custom layers)

# The CNN models bake in a Lambda(preprocess_input). Keras 3 saved that NAMED function
# by reference, so we must supply it at load time via custom_objects, using the SAME
# preprocess_input the model was built with. (ViT uses an anonymous lambda -> nothing to supply.)
PREPROCESS_MODULE = {
    MOBILENET:      "mobilenet",
    EFFICIENTNETB0: "efficientnet",
    EFFICIENTNETB3: "efficientnet",
    DENSENET121:    "densenet",
    VIT_B16:        None,
}

def get_custom_objects():
    mod_name = PREPROCESS_MODULE.get(CHOSEN_MODEL)
    if mod_name is None:
        return {}
    mod = importlib.import_module(f"tensorflow.keras.applications.{mod_name}")
    return {"preprocess_input": mod.preprocess_input}

def resolve_model_path(cfg):
    """Use the configured path if it exists; otherwise auto-find the file anywhere
    under /kaggle/input so the exact dataset mount path does not matter."""
    p = cfg["model_path"]
    if os.path.exists(p):
        return p
    hits = sorted(glob.glob(f"/kaggle/input/**/{cfg['filename']}", recursive=True))
    if hits:
        print(f"[info] '{p}' not found; using discovered path: {hits[0]}")
        return hits[0]
    print(f"Could not find {cfg['filename']} under /kaggle/input. Available .keras files:")
    for f in sorted(glob.glob("/kaggle/input/**/*.keras", recursive=True)):
        print("   ", f)
    raise FileNotFoundError(f"{cfg['filename']} not found. Fix MODELS_DIR or the uploaded dataset.")

def load_any(path):
    # custom_objects supplies preprocess_input; safe_mode=False allows Lambda deserialize.
    custom = get_custom_objects()
    try:
        return tf.keras.models.load_model(path, compile=False, safe_mode=False, custom_objects=custom)
    except TypeError:
        return tf.keras.models.load_model(path, compile=False, custom_objects=custom)

resolved_path = resolve_model_path(CFG)
model = load_any(resolved_path)
print(f"Loaded {CFG['name']} from {resolved_path}")
print("Model output shape:", model.output_shape, "(expected (None, 3))")

## 5. Build the test dataset

In [ ]:
# =========================================================
# 5. TEST DATASET  (identical for all models: feed [0,1] images)
# =========================================================
test_meta = pd.read_csv(TEST_CSV_PATH)
unique_test_images = test_meta[["image_path"]].drop_duplicates().reset_index(drop=True)

def preprocess_test_img(img_path):
    full_path = tf.strings.join([KAGGLE_PATH, "/", img_path])
    img = tf.io.read_file(full_path)
    img = tf.image.decode_jpeg(img, channels=3)
    img = tf.image.resize(img, [IMG_SIZE, IMG_SIZE])
    img = tf.cast(img, tf.float32) / 255.0        # [0,1] -> model rescales to [0,255]/preprocess internally
    return img

test_ds = (tf.data.Dataset
           .from_tensor_slices(unique_test_images["image_path"].values.astype(str))
           .map(preprocess_test_img, num_parallel_calls=tf.data.AUTOTUNE)
           .batch(BATCH_SIZE)
           .prefetch(tf.data.AUTOTUNE))
print(f"{len(unique_test_images)} unique test images")

## 6. Inference (raw grams — no expm1)

In [ ]:
# =========================================================
# 6. INFERENCE  --  models output RAW grams (USE_LOG_TARGETS=False)
# =========================================================
print("Running inference ...")
raw_preds = model.predict(test_ds, verbose=1)     # shape (N, 3) = [Green, Dead, Clover]
raw_preds = np.maximum(raw_preds, 0.0)            # clip negatives  (NO expm1!)

pred_green  = raw_preds[:, 0]
pred_dead   = raw_preds[:, 1]
pred_clover = raw_preds[:, 2]

## 7. Auto-detect the GDM formula, then reconstruct GDM & Total

In [ ]:
# =========================================================
# 7. RECONSTRUCT GDM & TOTAL
# =========================================================
# Total_g is unambiguous: Green + Dead + Clover.
# GDM_g definition is auto-detected from train.csv (Green+Clover vs Green+Dead),
# so we never guess. Falls back to Green+Clover (standard "Green Dry Matter").
GDM_IS_GREEN_PLUS_CLOVER = True
try:
    tr = pd.read_csv(TRAIN_CSV_PATH)
    p = tr.pivot_table(index="image_path", columns="target_name", values="target")
    if {"GDM_g", "Dry_Green_g", "Dry_Clover_g", "Dry_Dead_g"}.issubset(p.columns):
        err_gc = (p["GDM_g"] - (p["Dry_Green_g"] + p["Dry_Clover_g"])).abs().mean()
        err_gd = (p["GDM_g"] - (p["Dry_Green_g"] + p["Dry_Dead_g"])).abs().mean()
        GDM_IS_GREEN_PLUS_CLOVER = err_gc <= err_gd
        print(f"GDM auto-detect -> Green+Clover err={err_gc:.4f} | Green+Dead err={err_gd:.4f}")
except Exception as e:
    print("GDM auto-detect skipped:", e)

if GDM_IS_GREEN_PLUS_CLOVER:
    pred_gdm = pred_green + pred_clover
    print("Using GDM = Green + Clover")
else:
    pred_gdm = pred_green + pred_dead
    print("Using GDM = Green + Dead")

pred_total = pred_green + pred_dead + pred_clover

pred_df = pd.DataFrame({
    "image_path":   unique_test_images["image_path"],
    "Dry_Green_g":  pred_green,
    "Dry_Dead_g":   pred_dead,
    "Dry_Clover_g": pred_clover,
    "GDM_g":        pred_gdm,
    "Dry_Total_g":  pred_total,
})
pred_df.head()

## 8. Format into the competition submission

In [ ]:
# =========================================================
# 8. FORMAT SUBMISSION  (long format, matched to test.csv ordering)
# =========================================================
melted = pred_df.melt(
    id_vars=["image_path"],
    value_vars=["Dry_Green_g", "Dry_Dead_g", "Dry_Clover_g", "GDM_g", "Dry_Total_g"],
    var_name="target_name",
    value_name="target",
)

# Merge onto test.csv so only the requested (sample_id, target_name) rows are kept, in order.
submission = pd.merge(
    test_meta[["sample_id", "image_path", "target_name"]],
    melted,
    on=["image_path", "target_name"],
    how="left",
)
final_submission = submission[["sample_id", "target"]].copy()
final_submission.head(10)

## 9. Integrity checks & save

In [ ]:
# =========================================================
# 9. VERIFY & SAVE
# =========================================================
sample_sub = pd.read_csv(SAMPLE_SUB_PATH)
assert len(final_submission) == len(sample_sub), \
    f"Row count mismatch! expected {len(sample_sub)}, got {len(final_submission)}"
assert not final_submission["target"].isnull().any(), "NaN values found in predictions!"
assert (final_submission["target"] >= 0.0).all(), "Negative values found in predictions!"

final_submission.to_csv("submission.csv", index=False)
print(f"submission.csv written for {CFG['name']}  ({len(final_submission)} rows) - all checks passed")
print(final_submission.head(10))